## Simulating traffic around a lock (with hydrodynamics)
In this notebook, we simulate a lock on a network which two opposingly directed vessels have to pass. We add a pre-coded complex lock object on the graph. Vessels are locked together if they can fit inside the lock, and arrive within the clustering time window. Changing water levels are implemented, which influences the duration of the levelling.

In [53]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
from opentnsim.lock.calculations import calculate_levelling_time, calculate_water_exchange_fluxes, calculate_aggregated_water_exchange_fluxes
from opentnsim.environment.mixins.hydrodynamics import HydrodynamicData

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# packege(s) needed to create hydrodynamic data
import pyzsf
import xarray as xr

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


In [2]:
%load_ext autoreload
%autoreload 2

#### 0. Create environment

In [3]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
simulation_stop = datetime.datetime(2025, 1, 3, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.simulation_start = simulation_start
env.simulation_stop = simulation_stop
env.epoch = simulation_start

#### 1. Create graph

In [4]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('Sea -1',geometry=transform(wgs84eqd_to_wgs84rad,Point(-350600,0)))
graph.add_node('Sea 0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('Canal 0',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('Canal 1',geometry=transform(wgs84eqd_to_wgs84rad,Point(350600,0)))

# add edges
graph.add_edge('Sea -1','Sea 0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-350600, 0),Point(-5000, 0)])), weight=1, length_m=350600-5000)
graph.add_edge('Sea 0','Sea -1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-350600, 0)])), weight=1, length_m=350600-5000)
graph.add_edge('Sea 0','Canal 0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('Canal 0','Sea 0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)
graph.add_edge('Canal 0','Canal 1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(350600, 0)])), weight=1, length_m=350600-5000)
graph.add_edge('Canal 1','Canal 0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(350600, 0),Point(5000, 0)])), weight=1, length_m=350600-5000)

# add graph to environment
env.graph = graph

In [5]:
graph_module.plot_graph(graph)

#### 1+ Adding environmental data

In [6]:
hydrodynamic_data = xr.Dataset()

stations = ['Canal 0', 'Sea 0']
time = pd.date_range(start = simulation_start,
                     end = simulation_stop,
                     freq = pd.Timedelta(minutes=5))


# Water level
static_water_level = np.zeros(len(time))
tidal_amplitude = 1.0
tidal_period = 12.5 #hours
tidal_water_level = [tidal_amplitude*np.sin(2*np.pi*(t-simulation_start).total_seconds()/(tidal_period*3600)) for t in time]
water_level_data = xr.DataArray(data=[static_water_level,tidal_water_level],coords={'STATION':stations,'TIME':time})
hydrodynamic_data['Water level'] = water_level_data

# Salinity
static_salinity_sea = 25.0*np.ones(len(time))
static_salinity_canal = 5.0*np.ones(len(time))
salinity_data = xr.DataArray(data=[static_salinity_canal,static_salinity_sea],coords={'STATION':stations,'TIME':time})
hydrodynamic_data['Salinity'] = salinity_data

# Water temperature
static_temperature_sea = 15.0*np.ones(len(time))
static_temperature_canal = 15.0*np.ones(len(time))
temperature_data = xr.DataArray(data=[static_temperature_canal,static_temperature_sea],coords={'STATION':stations,'TIME':time})
hydrodynamic_data['Temperature'] = temperature_data

In [7]:
HydrodynamicData(env=env, hydrodynamic_data = hydrodynamic_data);

#### 1+ Adding infrastructure

In [8]:
lock_chamber = IsLockChamber(env=env,
                             lock_depth = 10,
                             name='Lock',
                             gate_open = 'Sea 0',
                             edge = ('Sea 0','Canal 0'),
                             geometry_m = Polygon([Point(-200, -25),Point(-200, 25),Point(200, 25),Point(200, -25)]))

In [9]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('Sea 0','Canal 0'),
                                   distance_from_edge_start = 0)

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('Canal 0','Sea 0'),
                                   distance_from_edge_start = 0)

In [10]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = ['Sea 0','Canal 0'],
                             env=env,
                             name = 'Lock complex',)

#### 2. Create agents

In [11]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
    {}
)

In [12]:
def generate_vessel(
    env,
    name,
    start_node,
    end_node,
    arrival_time,
    vessel_speed=4,
    vessel_length=100,
    vessel_beam=20,
    vessel_draft=5,
    vessel_type="tanker"):
    """
    Creates and returns a Vessel object with a computed route through the environment graph.

    Parameters:
    ----------
    env : Environment
        The simulation environment containing the graph and other context.
    name : str
        Human readabile identifier for the vessel.
    start_node : str or int
        The starting node in the graph (converted to string).
    end_node : str or int
        The destination node in the graph (converted to string).
    arrival_time : pd.Timestamp
        The scheduled arrival time of the vessel at the start node.
    vessel_speed : float, optional
        Speed of the vessel in knots or simulation units (default is 4).
    vessel_length : float, optional
        Length of the vessel in meters (default is 100).
    vessel_beam : float, optional
        Beam (width) of the vessel in meters (default is 20).
    vessel_draft : float, optional
        Draught (depth below waterline) of the vessel in meters (default is 10).
    vessel_type : str, optional
        Type of vessel (e.g., "tanker", "cargo", "container") (default is "tanker").

    Returns:
    -------
    Vessel or None
        A Vessel object initialized with the given parameters and route.
        Returns None if no valid path exists between start_node and end_node.
    """
    
    # Ensure nodes are strings
    start_node = str(start_node)
    end_node = str(end_node)

    try:
        route = nx.dijkstra_path(env.graph, start_node, end_node)
    except nx.NetworkXNoPath:
        print(f"⚠️ No path from {start_node} to {end_node}. Vessel {name} not created.")
        return None

    geometry = env.graph.nodes[start_node]['geometry']

    data_vessel = {
        "env": env,
        "name": name,
        "geometry": geometry,
        "route": route,
        "v": vessel_speed,
        "L": vessel_length,
        "B": vessel_beam,
        "T": vessel_draft,
        "type": vessel_type,
        "arrival_time": arrival_time,
    }

    vessel = Vessel(**data_vessel)

    return vessel

In [13]:
def generate_vessels_with_distributions(
    env,
    num_vessels,
    start_time,
    arrival_dist_up=None,
    arrival_dist_down=None,
    seed_up=None,
    seed_down=None):
    """
    Generates a list of vessels with interarrival times drawn from specified distributions
    for upward and downward directions. Supports independent seeding for reproducibility.

    Parameters
    ----------
    env : Environment
        The simulation environment containing the graph and vessel context.
    num_vessels : int
        Total number of vessels to generate. Vessels alternate between up and down directions.
    start_time : pd.Timestamp
        The initial timestamp from which vessel arrivals begin.
    arrival_dist_up : callable, optional
        A function returning interarrival times (in minutes) for upward-moving vessels.
        If None, defaults to an exponential distribution with mean 20 minutes.
    arrival_dist_down : callable, optional
        A function returning interarrival times (in minutes) for downward-moving vessels.
        If None, defaults to an exponential distribution with mean 20 minutes.
    seed_up : int or None, optional
        Seed for the random number generator used in upward direction.
    seed_down : int or None, optional
        Seed for the random number generator used in downward direction.

    Returns
    -------
    list of Vessel
        A list of Vessel objects with assigned routes and arrival times.
        Vessels for which no valid path exists are skipped.
    """

    vessels = []

    # Create independent random generators
    rng_up = np.random.default_rng(seed_up)
    rng_down = np.random.default_rng(seed_down)

    # Default to exponential distribution with mean 20 minutes
    if arrival_dist_up is None:
        arrival_dist_up = lambda: rng_up.exponential(scale=20)
    if arrival_dist_down is None:
        arrival_dist_down = lambda: rng_down.exponential(scale=20)

    up_time = start_time
    down_time = start_time

    for i in range(num_vessels):
        if i % 2 == 0:
            # Upward direction: -1 → +1
            start_node, end_node = "Sea -1", "Canal 1"
            delta_minutes = arrival_dist_up()
            arrival_time = up_time + pd.Timedelta(minutes=delta_minutes)
            up_time = arrival_time
        else:
            # Downward direction: +1 → -1
            start_node, end_node = "Canal 1", "Sea -1"
            delta_minutes = arrival_dist_down()
            arrival_time = down_time + pd.Timedelta(minutes=delta_minutes)
            down_time = arrival_time

        vessel = generate_vessel(
            env=env,
            name=f"Vessel {i + 1}",
            start_node=start_node,
            end_node=end_node,
            arrival_time=arrival_time
        )

        if vessel:
            vessels.append(vessel)

    return vessels

In [14]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [15]:
# Example using numpy
# Create independent random generators
rng_up = np.random.default_rng(123)
rng_down = np.random.default_rng(456)

# Exponential distributions with different means (scale = mean)
arrival_dist_up = lambda: rng_up.exponential(scale=30)   # mean 30 minutes
arrival_dist_down = lambda: rng_down.exponential(scale=15)  # mean 15 minutes

vessels = generate_vessels_with_distributions(
    env=env,
    num_vessels=9,
    start_time=simulation_start,
    arrival_dist_up=arrival_dist_up,
    arrival_dist_down=arrival_dist_down,
    seed_up=123,
    seed_down=456
)

for vessel in vessels:
    env.process(mission(env, vessel))

#### 3. Run simulation

In [16]:
env.run()

#### 4. Inspect output

In [17]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_chamber.logbook)

print("'{}' logbook data:".format(lock_chamber.name))  
print('')

display(lock_df)

'Lock' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock gate closing start,2025-01-02 00:07:22.177207,{},Sea 0
1,Lock gate closing stop,2025-01-02 00:12:22.177207,{},Sea 0
2,Lock chamber converting start,2025-01-02 00:12:22.177207,{},Sea 0
3,Lock chamber converting stop,2025-01-02 00:27:52.177207,{},Canal 0
4,Lock gate opening start,2025-01-02 00:27:52.177207,{},Canal 0
5,Lock gate opening stop,2025-01-02 00:32:52.177207,{},Canal 0
6,Waiting for other vessels in lock start,2025-01-02 00:48:32.349993,{},Canal 0
7,Waiting for other vessels in lock stop,2025-01-02 00:52:40.773319,{},Canal 0
8,Lock gate closing start,2025-01-02 00:52:40.773319,{},Canal 0
9,Lock gate closing stop,2025-01-02 00:57:40.773319,{},Canal 0


#### Gantt chart of event table

In [19]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, lock_chamber])
fig = generate_vessel_gantt_chart(df_eventtable)

#### Time-distance diagram of vessels passing the lock and planning info

In [20]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chamber.plot(xlimmin = -6050, 
                        xlimmax = 6050,
                        ylimmin = pd.Timestamp('2025-01-01 22:00:00'),
                        ylimmax = pd.Timestamp('2025-01-02 09:00:00'),
                        method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

#### Saltwater intrusion

In [54]:
ZSF_results = calculate_water_exchange_fluxes(lock_chamber)
ZSF_results

C:\Users\floorbakker\Anaconda3\envs\opentnsim\lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning:

Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.



,head_sea,head_lake,routine,salinity_sea,salinity_lake,ship_volume_lake_to_sea,ship_volume_sea_to_lake,t_level,t_open_lake,t_open_sea,...,discharge_to_lake,discharge_to_sea,mass_transport_lake,mass_transport_sea,salinity_to_lake,salinity_to_sea,volume_from_lake,volume_from_sea,volume_to_lake,volume_to_sea
time,,,,,,,,,,,,,,,,,,,,,
2025-01-02 00:09:52.177207,-0.4067366430758012,0.0,4,25.0,5.0,0.0,0.0,0,0,86992.177207,...,0.000000e+00,2.205546,0.000000e+00,-1.918653e+06,15.000000,15.000000,0.000000,1.918653e+05,0.000000e+00,191865.267138
2025-01-02 00:12:22.177207,-0.4067366430758012,0.0,1,25.0,5.0,0.0,0.0,930.0,0,0,...,0.000000e+00,0.000000,4.067366e+04,0.000000e+00,25.000000,25.000000,8134.732862,0.000000e+00,0.000000e+00,0.000000
2025-01-02 00:30:22.177207,-0.2486898871648556,0.0,2,25.0,5.0,0.0,0.0,0,1488.596112,0,...,1.092786e+02,0.000000,-3.121104e+06,0.000000e+00,24.186527,24.186527,162671.626643,0.000000e+00,1.626716e+05,0.000000
2025-01-02 00:57:40.773319,1.2864981197413093e-15,0.0,3,25.0,5.0,0.0,0.0,10.0,0,0,...,0.000000e+00,0.000000,0.000000e+00,-6.432491e-10,8.581009,8.581009,0.000000,2.572996e-11,0.000000e+00,0.000000
2025-01-02 01:00:20.773319,1.2864981197413093e-15,0.0,4,25.0,5.0,0.0,0.0,0,0,1727.192224,...,0.000000e+00,97.240710,0.000000e+00,-2.757625e+06,8.581009,8.581009,0.000000,1.679534e+05,0.000000e+00,167953.398979
2025-01-02 01:31:37.965543,0.24868988716485466,0.0,1,25.0,5.0,0.0,0.0,670.0,0,0,...,3.840293e-14,0.000000,-5.755570e-10,0.000000e+00,22.369136,22.369136,0.000000,0.000000e+00,2.572996e-11,0.000000
2025-01-02 01:45:17.965543,0.36812455268467714,0.0,2,25.0,5.0,0.0,0.0,0,1138.596113,0,...,1.191866e+02,0.000000,-2.357085e+06,0.000000e+00,22.369136,22.369136,135705.349759,0.000000e+00,1.357053e+05,0.000000
2025-01-02 02:06:46.561656,0.6534206039901052,0.0,3,25.0,5.0,0.0,0.0,1080.0,0,0,...,0.000000e+00,0.000000,0.000000e+00,-3.267103e+05,10.583713,10.583713,0.000000,1.306841e+04,0.000000e+00,0.000000
2025-01-02 02:27:16.561656,0.6534206039901052,0.0,4,25.0,5.0,0.0,0.0,0,0,1115.172786,...,0.000000e+00,119.868130,0.000000e+00,-1.808882e+06,11.467926,11.467926,0.000000,1.336737e+05,0.000000e+00,133673.675983


In [62]:
ZSF_results_aggragated = calculate_aggregated_water_exchange_fluxes(lock_chamber,ZSF_results)
ZSF_results_aggragated

{'mass_transport_lake': -8689849.35713382,
 'mass_transport_sea': -6811870.350056563,
 'volume_from_lake': 506511.7092636549,
 'volume_from_sea': 506560.75418055395,
 'volume_to_lake': 511445.388481941,
 'volume_to_sea': 493492.3421007518,
 'salinity_to_sea': 11.85864096602842,
 'salinity_to_lake': 21.942534151617156,
 'discharge_from_lake': 2.9312020212017065,
 'discharge_from_sea': 2.9314858459522797,
 'discharge_to_lake': 2.9597534055667882,
 'discharge_to_sea': 2.8558584612312026}